<a href="https://colab.research.google.com/github/lsmee/Dimensions-Departmental-Reporting/blob/main/Departmental_Report_Part_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%javascript
function KeepAlive() {
  document.querySelector("#top-toolbar > colab-connect-button")
    .shadowRoot.querySelector("#connect").click();
}
setInterval(KeepAlive, 60000);

<IPython.core.display.Javascript object>

In [ ]:

print("Setting up environment...")
!pip install dimcli xlsxwriter pandas==2.2.2 scipy --quiet

import dimcli
import pandas as pd
import numpy as np
import time
import sys
import json
import ast
import xlsxwriter
import re
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.stats import gmean
from google.colab import files
from datetime import date as _date

# ==========================================================
# 1. UTILITIES
# ==========================================================

def safe_dsl_query(dsl, query, retries=3, delay=10):
    for i in range(retries):
        try:
            return dsl.query(query)
        except Exception as e:
            if any(code in str(e) for code in ["502", "503", "504"]):
                print(f"  ⚠️  Server busy. Retrying in {delay}s... ({i+1}/{retries})")
                time.sleep(delay)
            else:
                raise e
    return dsl.query(query)


def run_robust_query(dsl, query_template, ids, label="Data"):
    BATCH_SIZE = 25
    all_data = []
    total_batches = (len(ids) + BATCH_SIZE - 1) // BATCH_SIZE
    for i in range(0, len(ids), BATCH_SIZE):
        chunk = ids[i:i + BATCH_SIZE]
        ids_str = '"' + '", "'.join(chunk) + '"'
        query = query_template.replace("ID_BATCH", ids_str)
        time.sleep(0.3)
        print(f"  {label}: batch {i // BATCH_SIZE + 1}/{total_batches}...", end="\r")
        res = safe_dsl_query(dsl, query)
        if not res.errors:
            df = res.as_dataframe()
            if not df.empty:
                all_data.append(df)
        elif "code: 3" in str(res.errors):
            for rid in chunk:
                time.sleep(0.5)
                res_s = safe_dsl_query(dsl, query_template.replace("ID_BATCH", f'"{rid}"'))
                if not res_s.errors:
                    df_s = res_s.as_dataframe()
                    if not df_s.empty:
                        all_data.append(df_s)
    print(f"  ✅ {label}: done.                    ")
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()


def fetch_all_publications(dsl, researcher_ids, year_s, year_e):
    all_pubs = []
    PAGE_SIZE = 1000

    CORE_FIELDS = (
        'id+title+year+source_title+times_cited+field_citation_ratio'
        '+altmetric+type+doi+researchers+authors_count+open_access+journal'
    )
    EXTRA_FIELDS = (
        'id+category_for+research_org_countries+research_orgs'
        '+research_org_types+funders'
    )
    ALL_FIELDS = (
        'id+title+year+source_title+times_cited+field_citation_ratio'
        '+altmetric+category_for+type+doi+researchers'
        '+research_org_countries+research_orgs+research_org_types'
        '+authors_count+open_access+journal+funders'
    )

    def fetch_single_split(rid):
        print(f"\n  ⚠️  Using split-field fallback for researcher {rid}...")
        all_pages_core  = []
        all_pages_extra = []

        for fields, collector in [
            (CORE_FIELDS,  all_pages_core),
            (EXTRA_FIELDS, all_pages_extra),
        ]:
            skip = 0
            while True:
                query = f"""
                search publications
                where researchers = "{rid}"
                and year in [{year_s}:{year_e}]
                return publications[{fields}]
                limit {PAGE_SIZE} skip {skip}
                """
                time.sleep(0.2)
                res = safe_dsl_query(dsl, query)
                if res.errors:
                    print(f"\n  ⚠️  Split-fetch error for {rid} (DEBUG: {repr(res.errors)})")
                    break
                df_page = res.as_dataframe()
                if df_page.empty:
                    break
                collector.append(df_page)
                if len(df_page) < PAGE_SIZE:
                    break
                skip += PAGE_SIZE

        if not all_pages_core:
            return

        df_core = pd.concat(all_pages_core, ignore_index=True)
        if all_pages_extra:
            df_extra = pd.concat(all_pages_extra, ignore_index=True)
            extra_cols = [c for c in df_extra.columns if c != 'id']
            df_merged = df_core.merge(df_extra[['id'] + extra_cols], on='id', how='left')
        else:
            df_merged = df_core

        all_pubs.append(df_merged)
        print(f"  ✅ Split-field fallback complete for {rid} ({len(df_merged)} pubs).")

    def fetch_batch_full(chunk, skip=0):
        ids_str = '"' + '", "'.join(chunk) + '"'
        query = f"""
        search publications
        where researchers in [{ids_str}]
        and year in [{year_s}:{year_e}]
        return publications[{ALL_FIELDS}]
        limit {PAGE_SIZE} skip {skip}
        """
        time.sleep(0.1)
        return safe_dsl_query(dsl, query)

    def process_ids(ids, batch_size):
        if batch_size == 0:
            for rid in ids:
                fetch_single_split(rid)
            return

        total = (len(ids) + batch_size - 1) // batch_size
        for i in range(0, len(ids), batch_size):
            chunk = ids[i:i + batch_size]
            batch_num = i // batch_size + 1
            print(f"  Publications: batch {batch_num}/{total} (size={batch_size})...", end="\r")

            skip = 0
            batch_failed = False
            while True:
                res = fetch_batch_full(chunk, skip=skip)
                if res.errors:
                    err_str = str(res.errors).lower()
                    print(f"\n  DEBUG error repr: {repr(res.errors)[:200]}")
                    is_too_long = (
                        "code: 3" in err_str or
                        '"code": 3' in err_str or
                        "'code': 3" in err_str or
                        "query is too long" in err_str or
                        "too many filters" in err_str
                    )
                    if is_too_long:
                        if batch_size == 1:
                            new_size = 0
                        else:
                            new_size = max(batch_size // 2, 1)
                        print(f"\n  ⚠️  Query too long (size={batch_size}), retrying at size={new_size if new_size > 0 else 'split'}...")
                        process_ids(chunk, new_size)
                        batch_failed = True
                        break
                    print(f"\n  ⚠️  Unexpected error batch {batch_num}: {res.errors}")
                    break
                df_page = res.as_dataframe()
                if df_page.empty:
                    break
                all_pubs.append(df_page)
                if len(df_page) < PAGE_SIZE:
                    break
                skip += PAGE_SIZE

    process_ids(researcher_ids, batch_size=25)
    print(f"  ✅ Publications: done.                              ")
    return pd.concat(all_pubs, ignore_index=True) if all_pubs else pd.DataFrame()


def fetch_researcher_citation_counts(dsl, researcher_ids, year_s, year_e):
    print(f"\n  📊 Fetching per-researcher citation counts ({len(researcher_ids)} researchers)...")
    citation_counts = {}
    total = len(researcher_ids)
    PAGE_SIZE = 1000

    for i, rid in enumerate(researcher_ids):
        print(f"  Per-researcher citations: {i+1}/{total}...", end="\r")
        pub_ids_seen = set()
        total_cit = 0
        skip = 0
        while True:
            try:
                time.sleep(0.2)
                res = safe_dsl_query(dsl, f"""
                    search publications
                    where researchers = "{rid}"
                    and year in [{year_s}:{year_e}]
                    return publications[id+times_cited]
                    limit {PAGE_SIZE} skip {skip}
                """)
                if res.errors:
                    break
                df_page = res.as_dataframe()
                if df_page.empty:
                    break
                for _, row in df_page.iterrows():
                    if row['id'] not in pub_ids_seen:
                        pub_ids_seen.add(row['id'])
                        total_cit += int(row.get('times_cited', 0) or 0)
                if len(df_page) < PAGE_SIZE:
                    break
                skip += PAGE_SIZE
            except Exception as e:
                print(f"\n  ⚠️  Citation query error for {rid}: {e}")
                break
        citation_counts[rid] = total_cit

    print(f"  ✅ Per-researcher citations: done for {len(citation_counts)} researchers.          ")
    return citation_counts


def fetch_journal_metrics(dsl, journal_ids):
    JOURNAL_BATCH = 20
    all_journal_data = []
    total_batches = (len(journal_ids) + JOURNAL_BATCH - 1) // JOURNAL_BATCH
    print(f"  Fetching metrics for {len(journal_ids)} unique journals...")
    for i in range(0, len(journal_ids), JOURNAL_BATCH):
        chunk = journal_ids[i:i + JOURNAL_BATCH]
        ids_str = '"' + '", "'.join(chunk) + '"'
        query = f'search source_titles where id in [{ids_str}] return source_titles[id+snip+sjr]'
        time.sleep(0.3)
        print(f"  Journals: batch {i // JOURNAL_BATCH + 1}/{total_batches}...", end="\r")
        res = safe_dsl_query(dsl, query)
        if not res.errors:
            df_batch = res.as_dataframe()
            if not df_batch.empty:
                all_journal_data.append(df_batch)
        else:
            for jid in chunk:
                time.sleep(0.3)
                res_s = safe_dsl_query(dsl, f'search source_titles where id in ["{jid}"] return source_titles[id+snip+sjr]')
                if not res_s.errors:
                    df_s = res_s.as_dataframe()
                    if not df_s.empty:
                        all_journal_data.append(df_s)
    print(f"  ✅ Journals: done.                    ")
    return pd.concat(all_journal_data, ignore_index=True) if all_journal_data else pd.DataFrame()


def get_patent_citations(pub_ids, dsl):
    if not pub_ids:
        return {}, []
    print(f"  📄 Fetching patent citations for {len(pub_ids)} publications...")
    raw_patents = []
    BATCH_SIZE = 500
    total_batches = (len(pub_ids) + BATCH_SIZE - 1) // BATCH_SIZE
    pub_id_set = set(pub_ids)
    for i in range(0, len(pub_ids), BATCH_SIZE):
        batch = pub_ids[i:i + BATCH_SIZE]
        query = f"""
        search patents
        where publication_ids in {json.dumps(batch)}
        and legal_status in ["Granted", "Active"]
        return patents[id+publication_ids+assignee_names+assignee_countries]
        limit 1000
        """
        try:
            time.sleep(0.3)
            res = safe_dsl_query(dsl, query)
            print(f"  Patent citations: batch {i // BATCH_SIZE + 1}/{total_batches}...", end="\r")
            if res and not res.errors:
                for patent in res.patents:
                    patent_id  = patent.get('id', '')
                    cited_pubs = [p for p in (patent.get('publication_ids') or []) if p in pub_id_set]
                    assignee_names = patent.get('assignee_names') or []
                    ac_raw = patent.get('assignee_countries') or []
                    assignee_countries = [c.get('name', 'Unknown') if isinstance(c, dict) else str(c) for c in ac_raw]
                    max_len = max(len(assignee_names), len(assignee_countries), 1)
                    names_padded     = assignee_names     + ['Unknown'] * (max_len - len(assignee_names))
                    countries_padded = assignee_countries + ['Unknown'] * (max_len - len(assignee_countries))
                    for pub_id in cited_pubs:
                        raw_patents.append({'patent_id': patent_id, 'pub_id': pub_id,
                                            'assignees_paired': list(zip(names_padded, countries_padded))})
        except Exception as e:
            print(f"\n  ⚠️  Patent query error (batch {i // BATCH_SIZE + 1}): {e}")
    seen, deduped = set(), []
    for rec in raw_patents:
        key = (rec['patent_id'], rec['pub_id'])
        if key not in seen:
            seen.add(key)
            deduped.append(rec)
    patent_counts = defaultdict(int)
    for rec in deduped:
        patent_counts[rec['pub_id']] += 1
    patent_counts = dict(patent_counts)
    for pid in pub_ids:
        patent_counts.setdefault(pid, 0)
    assignee_rows = []
    for rec in deduped:
        for name, country in rec['assignees_paired']:
            assignee_rows.append({'patent_id': rec['patent_id'], 'assignee_name': name,
                                  'assignee_country': country, 'pub_id': rec['pub_id']})
    total = sum(patent_counts.values())
    pubs_with = sum(1 for v in patent_counts.values() if v > 0)
    print(f"  ✅ Patent citations (Granted/Active): {total} across {pubs_with} publications.")
    return patent_counts, assignee_rows


def get_policy_citations_and_publishers(pub_ids, dsl):
    if not pub_ids:
        return {}, []
    print(f"  📄 Fetching policy citations for {len(pub_ids)} publications...")
    raw_policy = []
    BATCH_SIZE = 500
    total_batches = (len(pub_ids) + BATCH_SIZE - 1) // BATCH_SIZE
    pub_id_set = set(pub_ids)
    for i in range(0, len(pub_ids), BATCH_SIZE):
        batch = pub_ids[i:i + BATCH_SIZE]
        query = f"""
        search policy_documents
        where publication_ids in {json.dumps(batch)}
        return policy_documents[id+publication_ids+publisher_org]
        limit 1000
        """
        try:
            time.sleep(0.3)
            res = safe_dsl_query(dsl, query)
            print(f"  Policy citations: batch {i // BATCH_SIZE + 1}/{total_batches}...", end="\r")
            if res and not res.errors:
                for doc in res.policy_documents:
                    doc_id = doc.get('id', '')
                    cited_pubs = [p for p in (doc.get('publication_ids') or []) if p in pub_id_set]
                    publisher_org = doc.get('publisher_org') or {}
                    publisher = publisher_org.get('name', 'Unknown') if isinstance(publisher_org, dict) else 'Unknown'
                    for pub_id in cited_pubs:
                        raw_policy.append({'doc_id': doc_id, 'pub_id': pub_id, 'publisher': publisher})
        except Exception as e:
            print(f"\n  ⚠️  Policy query error (batch {i // BATCH_SIZE + 1}): {e}")
    seen, deduped = set(), []
    for rec in raw_policy:
        key = (rec['doc_id'], rec['pub_id'])
        if key not in seen:
            seen.add(key)
            deduped.append(rec)
    policy_counts = defaultdict(int)
    for rec in deduped:
        policy_counts[rec['pub_id']] += 1
    policy_counts = dict(policy_counts)
    for pid in pub_ids:
        policy_counts.setdefault(pid, 0)
    publisher_rows = [{'publisher': rec['publisher'], 'pub_id': rec['pub_id']} for rec in deduped]
    total = sum(policy_counts.values())
    pubs_with = sum(1 for v in policy_counts.values() if v > 0)
    print(f"  ✅ Policy citations: {total} across {pubs_with} publications.")
    return policy_counts, publisher_rows


def get_clinical_trial_citations(pub_ids, dsl):
    if not pub_ids:
        return {}, []
    print(f"  📄 Fetching clinical trial citations for {len(pub_ids)} publications...")
    raw_trials = []
    BATCH_SIZE = 500
    total_batches = (len(pub_ids) + BATCH_SIZE - 1) // BATCH_SIZE
    pub_id_set = set(pub_ids)
    for i in range(0, len(pub_ids), BATCH_SIZE):
        batch = pub_ids[i:i + BATCH_SIZE]
        query = f"""
        search clinical_trials
        where publication_ids in {json.dumps(batch)}
        return clinical_trials[id+publication_ids+research_orgs+funder_countries]
        limit 1000
        """
        try:
            time.sleep(0.3)
            res = safe_dsl_query(dsl, query)
            print(f"  Clinical trial citations: batch {i // BATCH_SIZE + 1}/{total_batches}...", end="\r")
            if res and not res.errors:
                for trial in res.clinical_trials:
                    trial_id   = trial.get('id', '')
                    cited_pubs = [p for p in (trial.get('publication_ids') or []) if p in pub_id_set]
                    research_orgs = trial.get('research_orgs') or []
                    org_names = [o.get('name', 'Unknown') if isinstance(o, dict) else str(o) for o in research_orgs] or ['Unknown']
                    funder_countries_raw = trial.get('funder_countries') or []
                    org_countries = [c.get('name', 'Unknown') if isinstance(c, dict) else str(c) for c in funder_countries_raw] or ['Unknown']
                    max_len = max(len(org_names), len(org_countries), 1)
                    names_padded     = org_names     + ['Unknown'] * (max_len - len(org_names))
                    countries_padded = org_countries + ['Unknown'] * (max_len - len(org_countries))
                    for pub_id in cited_pubs:
                        raw_trials.append({'trial_id': trial_id, 'pub_id': pub_id,
                                           'orgs_paired': list(zip(names_padded, countries_padded))})
        except Exception as e:
            print(f"\n  ⚠️  Clinical trial query error (batch {i // BATCH_SIZE + 1}): {e}")
    seen, deduped = set(), []
    for rec in raw_trials:
        key = (rec['trial_id'], rec['pub_id'])
        if key not in seen:
            seen.add(key)
            deduped.append(rec)
    clinical_counts = defaultdict(int)
    for rec in deduped:
        clinical_counts[rec['pub_id']] += 1
    clinical_counts = dict(clinical_counts)
    for pid in pub_ids:
        clinical_counts.setdefault(pid, 0)
    sponsor_rows = []
    for rec in deduped:
        for org_name, org_country in rec['orgs_paired']:
            sponsor_rows.append({'trial_id': rec['trial_id'], 'org_name': org_name,
                                 'org_country': org_country, 'pub_id': rec['pub_id']})
    total = sum(clinical_counts.values())
    pubs_with = sum(1 for v in clinical_counts.values() if v > 0)
    print(f"  ✅ Clinical trial citations: {total} across {pubs_with} publications.")
    return clinical_counts, sponsor_rows


def flatten_org_types(val):
    if isinstance(val, list):
        types = sorted(set(str(t).strip() for t in val if t))
        return ', '.join(types) if types else 'Unknown'
    if isinstance(val, str) and val.strip():
        return val.strip()
    return 'Unknown'


def get_latest_metric(val_list):
    if isinstance(val_list, list) and len(val_list) > 0:
        try:
            return sorted(val_list, key=lambda x: x.get('year', 0), reverse=True)[0].get('value')
        except:
            return np.nan
    return val_list if isinstance(val_list, (int, float)) else np.nan


def get_collab_safe(row):
    c = row.get('research_org_countries')
    o = row.get('research_orgs')
    c = c if isinstance(c, list) else []
    o = o if isinstance(o, list) else []
    authors = row.get('authors_count', 0)
    if len(c) > 1:
        return 'International Collaboration'
    if len(c) == 1 and len(o) > 1:
        return 'Only national collaboration'
    if len(o) == 1 and authors > 1:
        return 'Only institutional collaboration'
    return 'Single authorship (no collaboration)'


# ==========================================================
# 2. LOGIN
# ==========================================================
print("\n--- STEP 1: DIMENSIONS LOGIN ---")
api_key = input("Please paste your Dimensions API Key: ").strip()
try:
    dimcli.login(key=api_key)
    dsl = dimcli.Dsl()
    print("✅ Connected.")
except Exception as e:
    sys.exit(f"❌ Login failed: {e}")


# ==========================================================
# 3. INPUT
# ==========================================================
print("\n--- STEP 2: INPUT PARAMETERS ---")
YEAR_S = int(input("Start Year (e.g., 2020): "))
YEAR_E = int(input("End Year (e.g., 2025): "))
FACULTY = input("Faculty / Department name (e.g., Faculty of Science): ").strip()

GRID_ID = input("Institution GRID ID (e.g., grid.1005.4): ").strip()
print(f"  Looking up institution name for {GRID_ID}...")
try:
    res_grid = safe_dsl_query(dsl, f"""
        search organizations
        where id = "{GRID_ID}"
        return organizations[name]
    """)
    if not res_grid.errors and not res_grid.as_dataframe().empty:
        INSTITUTION = res_grid.as_dataframe().iloc[0]['name']
        print(f"  ✅ Institution found: {INSTITUTION}")
    else:
        print(f"  ⚠️  Could not find GRID ID '{GRID_ID}'.")
        INSTITUTION = input("  Please enter institution name manually: ").strip()
except Exception as e:
    print(f"  ⚠️  GRID lookup failed: {e}")
    INSTITUTION = input("  Please enter institution name manually: ").strip()

print(f"\n  Institution : {INSTITUTION}")
print(f"  Faculty     : {FACULTY}")
print(f"  Period      : {YEAR_S} – {YEAR_E}")

print("\nPlease upload your researcher ID list (CSV or Excel):")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df_input = pd.read_csv(filename) if filename.endswith('.csv') else pd.read_excel(filename)
RESEARCHER_IDS = df_input.iloc[:, 0].dropna().astype(str).unique().tolist()
print(f"✅ Loaded {len(RESEARCHER_IDS)} researcher IDs.")


# ==========================================================
# 4. FETCH DATA
# ==========================================================
print(f"\n--- STEP 3: FETCHING DATA ---")

print("\n[1/5] Fetching researcher metadata...")
q_meta = (
    "search researchers where id in [ID_BATCH] "
    "return researchers[id+first_name+last_name+orcid_id+current_research_org]"
)
df_res_meta = run_robust_query(dsl, q_meta, RESEARCHER_IDS, label="Metadata")

print("\n[2/5] Fetching publications (paginated, includes citations field)...")
df_pubs_raw = fetch_all_publications(dsl, RESEARCHER_IDS, YEAR_S, YEAR_E)

if df_pubs_raw.empty:
    print("❌ No publications found.")
    sys.exit()

print(f"  📚 {len(df_pubs_raw)} records fetched (before dedup).")
df_up = df_pubs_raw.drop_duplicates(subset='id').copy()
print(f"  📚 {len(df_up)} unique publications.")

if 'altmetric' not in df_up.columns:
    df_up['altmetric'] = None

all_pub_ids = df_up['id'].unique().tolist()

print("\n[3/5] Fetching per-researcher citation counts (one query per researcher)...")
researcher_citation_counts = fetch_researcher_citation_counts(dsl, RESEARCHER_IDS, YEAR_S, YEAR_E)

# ── PARALLEL FETCH: patent + policy + clinical ────────────────────────────────
print("\n[4/5] Fetching patent, policy, and clinical trial citations IN PARALLEL...")

_results = {}

def _run_patents():
    return ('patents', get_patent_citations(all_pub_ids, dsl))

def _run_policy():
    return ('policy', get_policy_citations_and_publishers(all_pub_ids, dsl))

def _run_clinical():
    return ('clinical', get_clinical_trial_citations(all_pub_ids, dsl))

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [
        executor.submit(_run_patents),
        executor.submit(_run_policy),
        executor.submit(_run_clinical),
    ]
    for future in as_completed(futures):
        key, result = future.result()
        _results[key] = result

patent_cit_counts,  assignee_rows  = _results['patents']
policy_cit_counts,  publisher_rows = _results['policy']
clinical_cit_counts, sponsor_rows  = _results['clinical']

print("  ✅ Parallel fetch complete.")

df_up['patent_citations']   = df_up['id'].map(patent_cit_counts).fillna(0).astype(int)
df_up['policy_citations']   = df_up['id'].map(policy_cit_counts).fillna(0).astype(int)
df_up['clinical_citations'] = df_up['id'].map(clinical_cit_counts).fillna(0).astype(int)

print("\n[5/5] Fetching journal metrics (SNIP / SJR)...")
df_journals = pd.DataFrame()
if 'journal.id' in df_up.columns:
    jids = df_up['journal.id'].dropna().unique().tolist()
    df_st = fetch_journal_metrics(dsl, jids)
    if not df_st.empty:
        df_st['SNIP'] = df_st['snip'].apply(get_latest_metric)
        df_st['SJR']  = df_st['sjr'].apply(get_latest_metric)
        df_journals = pd.merge(
            df_up.groupby(['journal.id', 'journal.title']).size().reset_index(name='Count'),
            df_st[['id', 'SNIP', 'SJR']],
            left_on='journal.id', right_on='id', how='left'
        ).drop(columns=['id'])
        filled = df_journals['SNIP'].notna().sum()
        print(f"  📊 SNIP/SJR populated for {filled}/{len(df_journals)} journals.")
else:
    print("  ⚠️  No journal.id column — skipping journal metrics.")


# ==========================================================
# 5. ANALYSIS
# ==========================================================
print("\n--- STEP 4: COMPUTING METRICS ---")

for_exp = df_up.explode('category_for').dropna(subset=['category_for'])
for_exp['Field_Name'] = for_exp['category_for'].apply(
    lambda x: x.get('name') if isinstance(x, dict) else str(x)
)
df_for = for_exp['Field_Name'].value_counts().reset_index()
df_for.columns = ['Field', 'Count']

df_up['Collaboration_Type'] = df_up.apply(get_collab_safe, axis=1)
df_up['Journal_Name']        = df_up.get('journal.title', 'N/A')
df_up['source_title.title']  = df_up.get('journal.title', 'N/A')

if 'research_org_types' in df_up.columns:
    df_up['Org_Collab_Types'] = df_up['research_org_types'].apply(flatten_org_types)
else:
    df_up['Org_Collab_Types'] = 'Unknown'
    print("  ⚠️  research_org_types not returned — Org_Collab_Types set to Unknown.")

if 'research_org_types' in df_up.columns:
    org_types_exp = df_up[['id', 'research_org_types']].copy()
    org_types_exp['research_org_types'] = org_types_exp['research_org_types'].apply(
        lambda x: x if isinstance(x, list) else (['Unknown'] if not x else [str(x)])
    )
    org_types_exp = org_types_exp.explode('research_org_types')
    org_types_exp['research_org_types'] = org_types_exp['research_org_types'].fillna('Unknown').str.strip()
    df_org_collab_summary = (
        org_types_exp.groupby('research_org_types')['id']
        .nunique().reset_index()
        .sort_values('id', ascending=False).reset_index(drop=True)
    )
    df_org_collab_summary.columns = ['Organisation Type', 'Publications']
else:
    df_org_collab_summary = pd.DataFrame(columns=['Organisation Type', 'Publications'])
print(f"  📊 {len(df_org_collab_summary)} org type rows built.")

# ── Industry Co-authorship ────────────────────────────────────────────────────
print("  📊 Building Industry Co-authorship tables...")
IND_COLS_ORG = ['Organisation', 'Co-authored Publications', 'Total Citations']
IND_COLS_PUB = ['Title', 'DOI', 'Year', 'Type', 'Co-authoring Organisations', 'Citations']

if 'research_orgs' in df_up.columns and 'research_org_types' in df_up.columns:
    def _has_company(types_val):
        if isinstance(types_val, list):
            return any('company' in str(t).lower() for t in types_val)
        return 'company' in str(types_val).lower()

    df_ind = df_up[df_up['research_org_types'].apply(_has_company)].copy()

    ind_org_rows = []
    for _, row in df_ind.iterrows():
        orgs  = row.get('research_orgs') or []
        types = row.get('research_org_types') or []
        if isinstance(orgs, list) and isinstance(types, list):
            for o in orgs:
                if isinstance(o, dict):
                    name  = o.get('name', 'Unknown')
                    otype = o.get('type', '') or ''
                    otypes = o.get('types', [otype] if otype else [])
                    if any('company' in str(t).lower() for t in otypes) or not otypes:
                        ind_org_rows.append({'org': name, 'pub_id': row['id'],
                                             'citations': int(row.get('times_cited', 0) or 0)})
        elif isinstance(orgs, list):
            for o in orgs:
                name = o.get('name', 'Unknown') if isinstance(o, dict) else str(o)
                ind_org_rows.append({'org': name, 'pub_id': row['id'],
                                     'citations': int(row.get('times_cited', 0) or 0)})

    if ind_org_rows:
        df_ind_orgs = pd.DataFrame(ind_org_rows)
        df_industry_orgs = (
            df_ind_orgs.groupby('org')
            .agg(Co_authored_Publications=('pub_id', 'nunique'),
                 Total_Citations=('citations', 'sum'))
            .reset_index()
            .sort_values('Co_authored_Publications', ascending=False)
            .reset_index(drop=True)
        )
        df_industry_orgs.columns = IND_COLS_ORG
    else:
        df_industry_orgs = pd.DataFrame(columns=IND_COLS_ORG)

    keep_cols = ['title', 'doi', 'year', 'type', 'research_orgs', 'times_cited']
    df_ind_pubs = df_ind[keep_cols].copy()
    df_ind_pubs['co_orgs'] = df_ind_pubs['research_orgs'].apply(
        lambda x: ', '.join([
            o.get('name', '') for o in x
            if isinstance(o, dict) and any(
                'company' in str(t).lower()
                for t in (o.get('types') or ([o.get('type','')] if o.get('type') else []))
            )
        ]) if isinstance(x, list) else ''
    )
    df_ind_pubs['times_cited'] = pd.to_numeric(df_ind_pubs['times_cited'], errors='coerce').fillna(0).astype(int)
    df_ind_pubs = df_ind_pubs.sort_values('times_cited', ascending=False).head(10).reset_index(drop=True)
    df_ind_pubs = df_ind_pubs[['title', 'doi', 'year', 'type', 'co_orgs', 'times_cited']]
    df_ind_pubs.columns = IND_COLS_PUB

    print(f"  ✅ Industry Co-authorship: {len(df_ind)} co-authored pubs, {len(df_industry_orgs)} unique companies.")
else:
    df_industry_orgs = pd.DataFrame(columns=IND_COLS_ORG)
    df_ind_pubs      = pd.DataFrame(columns=IND_COLS_PUB)
    print("  ⚠️  research_orgs/research_org_types not available — skipping industry co-authorship.")


if 'funders' in df_up.columns:
    df_up['funders'] = df_up['funders'].apply(
        lambda x: ', '.join([f.get('name', '') for f in x if isinstance(f, dict)])
        if isinstance(x, list) else ''
    )
else:
    df_up['funders'] = ''

# Researcher-level summary
df_res_stats = df_up.explode('researchers').copy()
df_res_stats['res_id'] = df_res_stats['researchers'].apply(
    lambda x: x.get('id') if isinstance(x, dict) else None
)
df_res_stats = df_res_stats[df_res_stats['res_id'].isin(RESEARCHER_IDS)]
df_res_stats = df_res_stats.drop_duplicates(subset=['res_id', 'id'])

df_stats = (
    df_res_stats
    .groupby('res_id')
    .agg(
        Total_pubs=('id', 'nunique'),
        Total_Patent_Citations=('patent_citations', 'sum'),
        Total_Policy_Citations=('policy_citations', 'sum'),
        Total_Clinical_Citations=('clinical_citations', 'sum'),
        Avg_Altmetric=('altmetric', lambda x: round(x.mean(), 2) if x.notna().any() else 0)
    )
    .reset_index()
    .rename(columns={'res_id': 'id'})
)
df_stats['Total_Citations'] = df_stats['id'].map(researcher_citation_counts).fillna(0).astype(int)
print(f"  📊 Citation counts mapped for {df_stats['Total_Citations'].gt(0).sum()} researchers.")

if not df_res_meta.empty:
    org_col = (
        'current_research_org.name'
        if 'current_research_org.name' in df_res_meta.columns
        else 'current_research_org'
    )
    df_res_meta['Affiliation'] = df_res_meta[org_col].apply(
        lambda x: x if isinstance(x, str) else (x.get('name') if isinstance(x, dict) else "")
    )
drop_cols = [c for c in df_res_meta.columns if 'current_research_org' in c]
df_researcher_tab = (
    pd.merge(df_res_meta, df_stats, on='id', how='left')
    .fillna(0)
    .drop(columns=drop_cols, errors='ignore')
)

def _count_orcids(val):
    if isinstance(val, (list, np.ndarray)):
        items = val.tolist() if isinstance(val, np.ndarray) else val
        return len([v for v in items if v and str(v).strip() not in ('', '0')])
    try:
        if pd.isna(val):
            return 0
    except (TypeError, ValueError):
        pass
    if val == 0 or str(val).strip() in ('', '0'):
        return 0
    try:
        lst = ast.literal_eval(str(val))
        if isinstance(lst, list):
            return len([x for x in lst if x and str(x).strip()])
        return 1
    except:
        return 1 if str(val).strip() else 0

df_researcher_tab['_orcid_n'] = df_researcher_tab['orcid_id'].apply(_count_orcids)
total_researchers = len(df_researcher_tab)
with_orcid        = int((df_researcher_tab['_orcid_n'] >= 1).sum())
no_orcid          = int((df_researcher_tab['_orcid_n'] == 0).sum())
with_two_orcid    = int((df_researcher_tab['_orcid_n'] == 2).sum())
df_researcher_tab.drop(columns=['_orcid_n'], inplace=True)

def _top_res(col, display_col, n=10):
    sub = df_researcher_tab[df_researcher_tab[col] > 0].copy()
    sub = sub.sort_values(col, ascending=False).head(n).reset_index(drop=True)
    sub['_name'] = sub['first_name'].astype(str) + ' ' + sub['last_name'].astype(str)
    return sub[['_name', col]].rename(columns={'_name': 'Researcher', col: display_col})

top_res_policy    = _top_res('Total_Policy_Citations',    'Policy Citations')
top_res_patent    = _top_res('Total_Patent_Citations',    'Patent Citations')
top_res_clinical  = _top_res('Total_Clinical_Citations',  'Clinical Trial Citations')
top_res_pubs      = _top_res('Total_pubs',                'Total Publications')
top_res_cited     = _top_res('Total_Citations',           'Total Citations')
print(f"  📊 Top researcher tables built.")

df_collab = df_up.groupby('Collaboration_Type').size().reset_index(name='Count')

_country_exploded = (
    df_up['research_org_countries'].dropna()
    .apply(lambda x: [c['name'] for c in x] if isinstance(x, list) else [])
    .explode().reset_index(drop=True)
)
_country_counts = _country_exploded.value_counts()
HOME_COUNTRY = _country_counts.index[0] if len(_country_counts) else ''
print(f"  🌍 Home country detected: {HOME_COUNTRY}")

_pub_countries = df_up['research_org_countries'].apply(
    lambda x: list({c['name'] for c in x if c['name'] != HOME_COUNTRY})
    if isinstance(x, list) else []
)
df_country_collab = (
    _pub_countries.explode().dropna()
    .loc[lambda s: s != '']
    .value_counts().reset_index()
)
df_country_collab.columns = ['Country', 'Publications']
print(f"  🌍 {len(df_country_collab)} collaborating countries found.")

_fund_exp = df_up[df_up['funders'] != ''][['funders', 'times_cited']].copy()
_fund_exp['funder_list'] = _fund_exp['funders'].str.split(', ')
_fund_exp = _fund_exp.explode('funder_list')
_fund_exp['funder_list'] = _fund_exp['funder_list'].str.strip()
_fund_exp = _fund_exp[_fund_exp['funder_list'] != '']
df_funders_summary = (
    _fund_exp.groupby('funder_list')
    .agg(Publications=('funders', 'count'), Total_Citations=('times_cited', 'sum'))
    .reset_index().sort_values('Publications', ascending=False).reset_index(drop=True)
)
df_funders_summary.columns = ['Funder', 'Publications', 'Total Citations']
print(f"  📊 {len(df_funders_summary)} unique funders found.")

if publisher_rows:
    df_pub_rows  = pd.DataFrame(publisher_rows)
    doc_counts   = df_pub_rows.groupby('publisher').size().reset_index(name='Policy_Documents')
    pubs_cited   = df_pub_rows.groupby('publisher')['pub_id'].nunique().reset_index(name='Publications_Cited')
    df_policy_publishers = (
        pd.merge(doc_counts, pubs_cited, on='publisher')
        .sort_values('Publications_Cited', ascending=False).reset_index(drop=True)
    )
    df_policy_publishers.columns = ['Publisher', 'Policy Documents', 'Publications Cited']
    print(f"  📊 {len(df_policy_publishers)} unique policy publishers found.")
else:
    df_policy_publishers = pd.DataFrame(columns=['Publisher', 'Policy Documents', 'Publications Cited'])
    print("  ⚠️  No policy publisher data found.")

PAT_COLS = ['Assignee', 'Country', 'Patent Citations', 'Publications Cited']
if assignee_rows:
    df_ar = pd.DataFrame(assignee_rows)
    df_assignee_detail = (
        df_ar.groupby(['assignee_name', 'assignee_country'])
        .agg(Patent_Citations=('patent_id', 'count'), Publications_Cited=('pub_id', 'nunique'))
        .reset_index().sort_values('Patent_Citations', ascending=False).reset_index(drop=True)
    )
    df_assignee_detail.columns = PAT_COLS
    df_pat_country = (
        df_ar.groupby('assignee_country')
        .agg(Patent_Citations=('patent_id', 'count'), Publications_Cited=('pub_id', 'nunique'))
        .reset_index().sort_values('Patent_Citations', ascending=False).reset_index(drop=True)
    )
    df_pat_country.insert(0, 'Assignee', '— COUNTRY TOTAL —')
    df_pat_country.columns = PAT_COLS
    blank_p = pd.DataFrame([['', '', '', '']], columns=PAT_COLS)
    hdr_p   = pd.DataFrame([['── COUNTRY TOTALS ──', '', '', '']], columns=PAT_COLS)
    df_patents_summary = pd.concat([df_assignee_detail, blank_p, hdr_p, df_pat_country], ignore_index=True)
else:
    df_patents_summary = pd.DataFrame(columns=PAT_COLS)
    print("  ⚠️  No patent assignee data found.")

CT_COLS = ['Sponsor / Collaborator', 'Country', 'Trial Citations', 'Publications Cited']
if sponsor_rows:
    df_sr = pd.DataFrame(sponsor_rows)
    df_org_detail = (
        df_sr.groupby(['org_name', 'org_country'])
        .agg(Trial_Citations=('trial_id', 'count'), Publications_Cited=('pub_id', 'nunique'))
        .reset_index().sort_values('Trial_Citations', ascending=False).reset_index(drop=True)
    )
    df_org_detail.columns = CT_COLS
    df_ct_country = (
        df_sr.groupby('org_country')
        .agg(Trial_Citations=('trial_id', 'count'), Publications_Cited=('pub_id', 'nunique'))
        .reset_index().sort_values('Trial_Citations', ascending=False).reset_index(drop=True)
    )
    df_ct_country.insert(0, 'Sponsor / Collaborator', '— COUNTRY TOTAL —')
    df_ct_country.columns = CT_COLS
    blank_ct = pd.DataFrame([['', '', '', '']], columns=CT_COLS)
    hdr_ct   = pd.DataFrame([['── COUNTRY TOTALS ──', '', '', '']], columns=CT_COLS)
    df_clinical_summary = pd.concat([df_org_detail, blank_ct, hdr_ct, df_ct_country], ignore_index=True)
else:
    df_clinical_summary = pd.DataFrame(columns=CT_COLS)
    print("  ⚠️  No clinical trial sponsor data found.")

print("  📊 Building Publications Summary...")
fcr_col = 'field_citation_ratio'
if fcr_col in df_up.columns:
    df_up[fcr_col] = pd.to_numeric(df_up[fcr_col], errors='coerce')
else:
    df_up[fcr_col] = np.nan

years_sorted = sorted(df_up['year'].dropna().unique().astype(int).tolist())

def pub_summary_col(subset):
    fcr_vals = subset[fcr_col].dropna()
    fcr_vals = fcr_vals[fcr_vals > 0]
    return {
        'Total Publications':                  len(subset),
        'Total Citations':                     int(subset['times_cited'].sum()),
        'Geometric mean Field Citation Ratio': round(float(gmean(fcr_vals)), 3) if len(fcr_vals) > 0 else np.nan,
        'Publications with Altmetric Score':   int(subset['altmetric'].notna().sum()) if 'altmetric' in subset.columns else 0,
        'Open Access Publications (oa_all: gold, hybrid, bronze, green)': int(subset['open_access'].apply(
            lambda x: 'oa_all' in x if isinstance(x, list) else (isinstance(x, str) and 'oa_all' in x)
        ).sum()) if 'open_access' in subset.columns else 0,
        'Closed Access Publications': int(subset['open_access'].apply(
            lambda x: 'closed' in x if isinstance(x, list) else (isinstance(x, str) and 'closed' in x)
        ).sum()) if 'open_access' in subset.columns else 0,
    }

overall_metrics = pub_summary_col(df_up)
year_metrics    = {yr: pub_summary_col(df_up[df_up['year'] == yr]) for yr in years_sorted}

rows_ps = []
for metric in list(overall_metrics.keys()):
    row = {'Metric': metric, 'Overall': overall_metrics[metric]}
    for yr in years_sorted:
        row[str(yr)] = year_metrics[yr][metric]
    rows_ps.append(row)
df_pub_summary = pd.DataFrame(rows_ps)
print(f"  ✅ Publications Summary built.")

impact_summary = pd.DataFrame([
    {'Metric': 'Total Publications',                              'Count': len(df_up)},
    {'Metric': 'Publications with Altmetric Score',               'Count': int(df_up['altmetric'].notna().sum()) if 'altmetric' in df_up.columns else 0},
    {'Metric': 'Mean Altmetric Score',                            'Count': round(df_up['altmetric'].mean(), 2) if ('altmetric' in df_up.columns and df_up['altmetric'].notna().any()) else 0},
    {'Metric': 'Publications Cited by Patents (Granted/Active)',  'Count': int((df_up['patent_citations'] > 0).sum())},
    {'Metric': 'Total Patent Citations (Granted/Active)',          'Count': int(df_up['patent_citations'].sum())},
    {'Metric': 'Unique Patent Assignees',                         'Count': int(pd.DataFrame(assignee_rows)['assignee_name'].nunique()) if assignee_rows else 0},
    {'Metric': 'Publications Cited by Policy Docs',               'Count': int((df_up['policy_citations'] > 0).sum())},
    {'Metric': 'Total Policy Citations',                          'Count': int(df_up['policy_citations'].sum())},
    {'Metric': 'Unique Policy Publishers',                        'Count': len(df_policy_publishers)},
    {'Metric': 'Publications Cited by Clinical Trials',           'Count': int((df_up['clinical_citations'] > 0).sum())},
    {'Metric': 'Total Clinical Trial Citations',                  'Count': int(df_up['clinical_citations'].sum())},
    {'Metric': 'Unique Clinical Trial Sponsors/Collaborators',    'Count': int(pd.DataFrame(sponsor_rows)['org_name'].nunique()) if sponsor_rows else 0},
    {'Metric': 'Publications with Industry Co-authors (Company)',  'Count': len(df_ind) if 'df_ind' in dir() else 0},
    {'Metric': 'Unique Industry Co-authoring Organisations',       'Count': len(df_industry_orgs)},
    {'Metric': 'Publications with Government Co-authors',
     'Count': int(df_up['research_org_types'].apply(
         lambda v: any('government' in str(t).lower() for t in v) if isinstance(v, list) else False
     ).sum()) if 'research_org_types' in df_up.columns else 0},
    {'Metric': 'Unique Government Co-authoring Organisations',
     'Count': int(df_up[df_up['research_org_types'].apply(
         lambda v: any('government' in str(t).lower() for t in v) if isinstance(v, list) else False
     )]['research_orgs'].explode().dropna().apply(
         lambda o: o.get('name','') if isinstance(o, dict) else str(o)
     ).nunique()) if 'research_org_types' in df_up.columns and 'research_orgs' in df_up.columns else 0},
    {'Metric': 'Publications with Healthcare Co-authors',
     'Count': int(df_up['research_org_types'].apply(
         lambda v: any(kw in str(t).lower() for t in v for kw in ['healthcare','medical']) if isinstance(v, list) else False
     ).sum()) if 'research_org_types' in df_up.columns else 0},
    {'Metric': 'Unique Healthcare Co-authoring Organisations',
     'Count': int(df_up[df_up['research_org_types'].apply(
         lambda v: any(kw in str(t).lower() for t in v for kw in ['healthcare','medical']) if isinstance(v, list) else False
     )]['research_orgs'].explode().dropna().apply(
         lambda o: o.get('name','') if isinstance(o, dict) else str(o)
     ).nunique()) if 'research_org_types' in df_up.columns and 'research_orgs' in df_up.columns else 0},
    {'Metric': 'Publications with Facility Co-authors',
     'Count': int(df_up['research_org_types'].apply(
         lambda v: any('facility' in str(t).lower() for t in v) if isinstance(v, list) else False
     ).sum()) if 'research_org_types' in df_up.columns else 0},
    {'Metric': 'Unique Facility Co-authoring Organisations',
     'Count': int(df_up[df_up['research_org_types'].apply(
         lambda v: any('facility' in str(t).lower() for t in v) if isinstance(v, list) else False
     )]['research_orgs'].explode().dropna().apply(
         lambda o: o.get('name','') if isinstance(o, dict) else str(o)
     ).nunique()) if 'research_org_types' in df_up.columns and 'research_orgs' in df_up.columns else 0},
    {'Metric': 'Publications with Nonprofit Co-authors',
     'Count': int(df_up['research_org_types'].apply(
         lambda v: any('nonprofit' in str(t).lower() for t in v) if isinstance(v, list) else False
     ).sum()) if 'research_org_types' in df_up.columns else 0},
    {'Metric': 'Unique Nonprofit Co-authoring Organisations',
     'Count': int(df_up[df_up['research_org_types'].apply(
         lambda v: any('nonprofit' in str(t).lower() for t in v) if isinstance(v, list) else False
     )]['research_orgs'].explode().dropna().apply(
         lambda o: o.get('name','') if isinstance(o, dict) else str(o)
     ).nunique()) if 'research_org_types' in df_up.columns and 'research_orgs' in df_up.columns else 0},
    {'Metric': 'Publications with Funders Listed',                'Count': int((df_up['funders'] != '').sum())},
    {'Metric': 'Unique Funders',                                  'Count': len(df_funders_summary)},
])

# ── Complete Publications ─────────────────────────────────────────────────────
# Includes 'researchers' so generate_report.py can compute first/last author metrics.
keep = [
    'id', 'title', 'authors_count', 'category_for', 'doi', 'open_access',
    'research_org_countries', 'research_orgs', 'research_org_types',
    'times_cited', 'type', 'year', 'source_title.title', 'altmetric',
    'field_citation_ratio', 'Journal_Name', 'Collaboration_Type',
    'Org_Collab_Types', 'funders',
    'patent_citations', 'policy_citations', 'clinical_citations',
    'researchers',
]
df_complete = df_up.reindex(columns=keep)


# ==========================================================
# 6. EXPORT WITH EMBEDDED CHARTS — VERSION 22.21
# ==========================================================
print("\n--- STEP 5: EXPORTING WITH CHARTS ---")

TOP_N = 10

def _ordinal(n):
    suffixes = {1:'st', 2:'nd', 3:'rd'}
    if 11 <= (n % 100) <= 13:
        return f"{n}th"
    return f"{n}{suffixes.get(n % 10, 'th')}"

today       = _date.today()
report_date = f"{_ordinal(today.day)} {today.strftime('%B %Y')}"

p1_file = f'Researcher_Metrics_Report_{YEAR_S}_{YEAR_E}.xlsx'
with pd.ExcelWriter(p1_file, engine='xlsxwriter') as writer:

    wb = writer.book
    ri_ws = wb.add_worksheet('Report Information')

    df_researcher_tab.to_excel(writer,         sheet_name='Researchers',             index=False)
    impact_summary.to_excel(writer,            sheet_name='Impact Summary',          index=False)
    df_pub_summary.to_excel(writer,            sheet_name='Publications Summary',    index=False)
    df_journals.to_excel(writer,               sheet_name='Journals Summary',        index=False)
    df_for[['Field','Count']].to_excel(writer, sheet_name='Fields of Research',      index=False)
    df_collab.to_excel(writer,                 sheet_name='Collaboration Summary',   index=False)
    df_country_collab.to_excel(writer,         sheet_name='Country Collaboration',   index=False)
    df_org_collab_summary.to_excel(writer,     sheet_name='Org Type Collaboration',  index=False)
    df_industry_orgs.to_excel(writer,          sheet_name='Industry Co-authorship',  index=False)
    df_funders_summary.to_excel(writer,        sheet_name='Funders Summary',         index=False)
    df_policy_publishers.to_excel(writer,      sheet_name='Policy Publishers',       index=False)
    df_patents_summary.to_excel(writer,        sheet_name='Patents Summary',         index=False)
    df_clinical_summary.to_excel(writer,       sheet_name='Clinical Trials Summary', index=False)
    df_complete.to_excel(writer,               sheet_name='Complete Publications',   index=False)

    df_altmetric_top = (df_up[df_up['altmetric'].notna() & (df_up['altmetric'] > 0)]
                        [['title','doi','year','type','altmetric']]
                        .sort_values('altmetric', ascending=False)
                        .head(TOP_N).reset_index(drop=True))
    df_altmetric_top.columns = ['Title','DOI','Year','Type','Altmetric Score']
    df_altmetric_top.to_excel(writer, sheet_name='Altmetric Attention',       index=False)
    pd.DataFrame().to_excel(writer,   sheet_name='Top Researchers by Impact', index=False)

    # ── Formats ───────────────────────────────────────────
    hdr_fmt        = wb.add_format({'bold': True, 'bg_color': '#1F497D', 'font_color': 'white',
                                    'border': 1, 'text_wrap': True, 'valign': 'vcenter'})
    cell_fmt       = wb.add_format({'border': 1, 'valign': 'vcenter'})
    num_fmt        = wb.add_format({'border': 1, 'num_format': '#,##0', 'valign': 'vcenter'})
    dec_fmt        = wb.add_format({'border': 1, 'num_format': '0.000', 'valign': 'vcenter'})
    ri_title_fmt   = wb.add_format({'bold': True, 'font_size': 16, 'font_color': '#1F3864',
                                    'bottom': 2, 'bottom_color': '#1F497D'})
    ri_section_fmt = wb.add_format({'bold': True, 'font_size': 12, 'font_color': 'white',
                                    'bg_color': '#1F497D', 'border': 1})
    ri_label_fmt   = wb.add_format({'bold': True, 'font_size': 10, 'font_color': '#1F3864',
                                    'border': 1, 'bg_color': '#DCE6F1'})
    ri_value_fmt   = wb.add_format({'font_size': 10, 'border': 1, 'text_wrap': True, 'valign': 'vcenter'})
    ri_note_fmt    = wb.add_format({'font_size': 9, 'italic': True, 'font_color': '#595959', 'text_wrap': True})
    ri_tabhdr_fmt  = wb.add_format({'bold': True, 'font_size': 10, 'font_color': 'white',
                                    'bg_color': '#1F497D', 'border': 1})
    ri_tabname_fmt = wb.add_format({'bold': True, 'font_size': 10, 'border': 1, 'bg_color': '#F2F2F2'})
    ri_tabdesc_fmt = wb.add_format({'font_size': 10, 'border': 1, 'text_wrap': True, 'valign': 'vcenter'})
    ri_wide_fmt    = wb.add_format({'bold': True, 'bg_color': '#1F3864', 'font_color': 'white',
                                    'border': 1, 'font_size': 12})
    stripe_fmt     = wb.add_format({'border': 1, 'valign': 'vcenter', 'bg_color': '#EAF0F7'})
    stripe_num_fmt = wb.add_format({'border': 1, 'num_format': '#,##0', 'valign': 'vcenter',
                                    'bg_color': '#EAF0F7'})

    def safe_float(v):
        try:
            f = float(v)
            return 0.0 if (f != f) else f
        except:
            return 0.0

    # ── Report Information ────────────────────────────────
    ri_ws.set_column(0, 0, 30)
    ri_ws.set_column(1, 1, 60)
    ri_ws.set_row(0, 36)
    ri_ws.merge_range('A1:B1', 'Researcher Metrics Report — Report Information', ri_title_fmt)
    ri_ws.merge_range('A3:B3', 'Report Details', ri_section_fmt)
    details = [
        ('Date Generated',        report_date),
        ('Institution',           INSTITUTION),
        ('Faculty / Department',  FACULTY if FACULTY else 'Not specified'),
        ('Reporting Period',       f"{YEAR_S} – {YEAR_E}"),
        ('Number of Researchers', str(len(RESEARCHER_IDS))),
        ('Total Publications',    str(len(df_up))),
        ('Data Source',           'Dimensions (dimensions.ai)'),
    ]
    for i, (label, value) in enumerate(details):
        ri_ws.set_row(3 + i, 18)
        ri_ws.write(3 + i, 0, label, ri_label_fmt)
        ri_ws.write(3 + i, 1, value, ri_value_fmt)

    ri_ws.merge_range(11, 0, 11, 1, 'Methodology Notes', ri_section_fmt)
    notes = [
        ('Patent Citations',
         'Only patents with legal_status "Granted" or "Active" are included. '
         'Citation counts represent the number of (patent, publication) links. '
         'Fetched in parallel with policy and clinical trial queries.'),
        ('Policy Citations',
         'Counts represent unique policy documents citing each publication. '
         'Cross-batch deduplication applied. Fetched in parallel with patent '
         'and clinical trial queries.'),
        ('Field Citation Ratio (FCR)',
         'Geometric mean FCR, consistent with Dimensions methodology. '
         'Only publications with FCR > 0 are included.'),
        ('Open Access',
         'Counted where the Dimensions "open_access" field includes "oa_all" '
         '(gold, hybrid, bronze, green). Closed = tagged "closed".'),
        ('Organisation Types',
         'Sourced from GRID via Dimensions. A publication may contribute to '
         'multiple type counts if authors span different org types.'),
        ('Collaboration Type',
         'International = >1 country; National = 1 country >1 org; '
         'Institutional = 1 org >1 author; Single = sole author.'),
        ('Clinical Trial Sponsors',
         'Sourced from research_orgs on clinical trial records. Country from '
         'funder_countries. Fetched in parallel with patent and policy queries.'),
        ('Researcher Citations',
         'Citation totals fetched one researcher at a time, scoped to the '
         'reporting year range. Prevents inflation from co-authored papers '
         'being counted multiple times across researchers.'),
        ('Authorship Position',
         'Researchers field included in Complete Publications sheet '
         'to enable first/last author detection in generate_report.py. '
         'Corresponding author detection is not included at department level.'),
    ]
    for i, (label, note) in enumerate(notes):
        ri_ws.set_row(12 + i, 45)
        ri_ws.write(12 + i, 0, label, ri_label_fmt)
        ri_ws.write(12 + i, 1, note,  ri_value_fmt)

    row_td = 12 + len(notes) + 2
    ri_ws.merge_range(row_td, 0, row_td, 1, 'Report Tabs', ri_section_fmt)
    row_td += 1
    ri_ws.write(row_td, 0, 'Tab Name',    ri_tabhdr_fmt)
    ri_ws.write(row_td, 1, 'Description', ri_tabhdr_fmt)
    row_td += 1
    tab_descriptions = [
        ('Researchers',               'Per-researcher summary: publication count, citations, patent/policy/clinical citation totals, average Altmetric score.'),
        ('Impact Summary',            'Dataset-level metrics: publications, citations, patent, policy, clinical trial citations, funders, open access.'),
        ('Publications Summary',      'Year-by-year breakdown: total publications, citations, geometric mean FCR, Altmetric, open access counts.'),
        ('Journals Summary',          'All journals with publication counts, SNIP, and SJR. Top-10 formatted table.'),
        ('Fields of Research',        'ANZSRC Fields of Research classifications. Top-10 bar charts for 2-digit and 4-digit FoR codes.'),
        ('Collaboration Summary',     'Geographic collaboration type breakdown: International, National, Institutional, Single Authorship.'),
        ('Country Collaboration',     f'Publications by collaborating country (home country excluded). Full table + top 20 bar chart.'),
        ('Org Type Collaboration',    'Publications by research organisation type (Education, Government, Company, etc.).'),
        ('Industry Co-authorship',    'Publications co-authored with Company-type organisations. Top partner organisations (bar chart) and top cited co-authored publications.'),
        ('Funders Summary',           f'All funders sorted by publication count. Top-{TOP_N} formatted table + bar chart.'),
        ('Policy Publishers',         'Policy document publishers citing the publications. Top-10 table + top cited publications.'),
        ('Patents Summary',           'Granted/Active patent assignees + country rollup + top cited publications.'),
        ('Clinical Trials Summary',   'Clinical trial sponsors/collaborators + country rollup + top cited publications.'),
        ('Complete Publications',     'Full publication-level dataset including researchers field for first/last author position analysis.'),
        ('Altmetric Attention',       f'Top {TOP_N} publications by Altmetric Attention Score, hyperlinked.'),
        ('Top Researchers by Impact', 'Top-10 researchers by policy, patent, clinical citations, publications, citations. ORCID adoption summary.'),
    ]
    for tab_name, tab_desc in tab_descriptions:
        ri_ws.set_row(row_td, 36)
        ri_ws.write(row_td, 0, tab_name, ri_tabname_fmt)
        ri_ws.write(row_td, 1, tab_desc, ri_tabdesc_fmt)
        row_td += 1
    ri_ws.set_row(row_td + 1, 30)
    ri_ws.merge_range(row_td + 1, 0, row_td + 1, 1,
        'Generated using the Dimensions DSL API. All data sourced from Dimensions '
        '(dimensions.ai) and subject to their terms of use.', ri_note_fmt)
    print("  ✅ Report Information tab written.")

    # ── Chart helpers ─────────────────────────────────────
    def col_chart(ws, sheet_name, labels, values, title,
                  anchor_row, anchor_col, data_col,
                  chart_w=500, chart_h=340, x_title='', y_title='', integer_axis=False):
        if not labels: return
        n = len(labels)
        ws.write(0, data_col,     '_label_')
        ws.write(0, data_col + 1, '_value_')
        for i, (lbl, val) in enumerate(zip(labels, values)):
            ws.write(i + 1, data_col,     str(lbl))
            ws.write(i + 1, data_col + 1, safe_float(val))
        chart = wb.add_chart({'type': 'column'})
        chart.add_series({'name': title,
                          'categories': [sheet_name, 1, data_col,     n, data_col],
                          'values':     [sheet_name, 1, data_col + 1, n, data_col + 1],
                          'fill': {'color': '#1F497D'}, 'gap': 60,
                          'data_labels': {'value': True, 'font': {'size': 8}}})
        chart.set_title({'name': title, 'name_font': {'bold': True, 'size': 11, 'color': '#1F3864'}})
        chart.set_legend({'none': True})
        chart.set_chartarea({'border': {'color': '#D9D9D9'}})
        chart.set_plotarea({'border': {'color': '#D9D9D9'}})
        x_ax = {'name': x_title, 'name_font': {'size': 9, 'bold': True}} if x_title else {}
        y_ax = {'name': y_title, 'name_font': {'size': 9, 'bold': True}} if y_title else {}
        if integer_axis: y_ax['major_unit'] = 1
        chart.set_x_axis(x_ax); chart.set_y_axis(y_ax)
        chart.set_size({'width': chart_w, 'height': chart_h})
        ws.insert_chart(anchor_row, anchor_col, chart, {'x_offset': 5, 'y_offset': 5})

    def bar_chart(ws, sheet_name, labels, values, title,
                  anchor_row, anchor_col, data_col,
                  chart_w=520, chart_h=340, x_title='', y_title='', integer_axis=False):
        if not labels: return
        n = len(labels)
        ws.write(0, data_col,     '_label_')
        ws.write(0, data_col + 1, '_value_')
        for i, (lbl, val) in enumerate(zip(labels, values)):
            ws.write(i + 1, data_col,     str(lbl))
            ws.write(i + 1, data_col + 1, safe_float(val))
        chart = wb.add_chart({'type': 'bar'})
        chart.add_series({'name': title,
                          'categories': [sheet_name, 1, data_col,     n, data_col],
                          'values':     [sheet_name, 1, data_col + 1, n, data_col + 1],
                          'fill': {'color': '#1F497D'}, 'gap': 60,
                          'data_labels': {'value': True, 'font': {'size': 8}}})
        chart.set_title({'name': title, 'name_font': {'bold': True, 'size': 11, 'color': '#1F3864'}})
        chart.set_legend({'none': True})
        chart.set_chartarea({'border': {'color': '#D9D9D9'}})
        chart.set_plotarea({'border': {'color': '#D9D9D9'}})
        x_ax = {'name': x_title, 'name_font': {'size': 9, 'bold': True}} if x_title else {}
        y_ax = {'name': y_title, 'name_font': {'size': 9, 'bold': True}} if y_title else {}
        if integer_axis: x_ax['major_unit'] = 1
        chart.set_x_axis(x_ax); chart.set_y_axis(y_ax)
        chart.set_size({'width': chart_w, 'height': chart_h})
        ws.insert_chart(anchor_row, anchor_col, chart, {'x_offset': 5, 'y_offset': 5})

    def pie_chart(ws, sheet_name, labels, values, title,
                  anchor_row, anchor_col, data_col, chart_w=500, chart_h=380):
        if not labels: return
        n = len(labels)
        ws.write(0, data_col,     '_label_')
        ws.write(0, data_col + 1, '_value_')
        for i, (lbl, val) in enumerate(zip(labels, values)):
            ws.write(i + 1, data_col,     str(lbl))
            ws.write(i + 1, data_col + 1, safe_float(val))
        chart = wb.add_chart({'type': 'pie'})
        chart.add_series({'name': title,
                          'categories': [sheet_name, 1, data_col,     n, data_col],
                          'values':     [sheet_name, 1, data_col + 1, n, data_col + 1],
                          'data_labels': {'percentage': True, 'category': True,
                                          'separator': '\n', 'font': {'size': 8}}})
        chart.set_title({'name': title, 'name_font': {'bold': True, 'size': 11, 'color': '#1F3864'}})
        chart.set_legend({'position': 'bottom', 'font': {'size': 8}})
        chart.set_chartarea({'border': {'color': '#D9D9D9'}})
        chart.set_size({'width': chart_w, 'height': chart_h})
        ws.insert_chart(anchor_row, anchor_col, chart, {'x_offset': 5, 'y_offset': 5})

    # ── Impact Summary ────────────────────────────────────
    ws_impact = writer.sheets['Impact Summary']
    ws_impact.set_column(0, 0, 48); ws_impact.set_column(1, 1, 14)
    ws_impact.write(0, 0, 'Metric', hdr_fmt)
    ws_impact.write(0, 1, 'Count',  hdr_fmt)
    for i, row in impact_summary.iterrows():
        ws_impact.write(i + 1, 0, str(row['Metric']), cell_fmt)
        v = row['Count']
        if isinstance(v, float) and v == int(v):
            ws_impact.write(i + 1, 1, int(v), num_fmt)
        elif isinstance(v, float):
            ws_impact.write(i + 1, 1, float(v), dec_fmt)
        else:
            ws_impact.write(i + 1, 1, v, num_fmt)

    # ── Publications Summary ──────────────────────────────
    ws_pub   = writer.sheets['Publications Summary']
    yr_cols  = [c for c in df_pub_summary.columns if c not in ('Metric', 'Overall')]
    all_cols = ['Metric', 'Overall'] + yr_cols
    ws_pub.set_column(0, 0, 48)
    ws_pub.set_column(1, 1 + len(yr_cols), 12)
    for ci, cn in enumerate(all_cols):
        ws_pub.write(0, ci, cn, hdr_fmt)
    for ri2, row in df_pub_summary.iterrows():
        ws_pub.write(ri2 + 1, 0, str(row['Metric']), cell_fmt)
        for ci, cn in enumerate(['Overall'] + yr_cols):
            v = row[cn]
            if pd.isna(v):
                ws_pub.write(ri2 + 1, ci + 1, '', cell_fmt)
            elif isinstance(v, float) and v != int(v):
                ws_pub.write(ri2 + 1, ci + 1, round(float(v), 3), dec_fmt)
            else:
                ws_pub.write(ri2 + 1, ci + 1, int(v) if not pd.isna(v) else '', num_fmt)
    gap_row = len(df_pub_summary) + 3
    pub_row = df_pub_summary[df_pub_summary['Metric'] == 'Total Publications']
    if not pub_row.empty and yr_cols:
        col_chart(ws_pub, 'Publications Summary',
                  yr_cols, [safe_float(pub_row[c].values[0]) for c in yr_cols],
                  'Publications by Year', anchor_row=gap_row, anchor_col=0, data_col=30,
                  x_title='Year', y_title='No. Publications')
    if 'type' in df_up.columns:
        tc = df_up['type'].value_counts().reset_index()
        tc.columns = ['Type', 'Count']
        col_chart(ws_pub, 'Publications Summary',
                  tc['Type'].tolist(), tc['Count'].tolist(),
                  'Publications by Type', anchor_row=gap_row, anchor_col=9, data_col=34,
                  x_title='Publication Type', y_title='No. Publications')

    # ── Journals Summary ──────────────────────────────────
    ws_jour = writer.sheets['Journals Summary']
    df_j = df_journals[['journal.title','Count','SNIP','SJR']].copy()
    df_j['Count'] = pd.to_numeric(df_j['Count'], errors='coerce')
    df_j['SNIP']  = pd.to_numeric(df_j['SNIP'],  errors='coerce')
    df_j['SJR']   = pd.to_numeric(df_j['SJR'],   errors='coerce')
    df_j_top = df_j.sort_values('Count', ascending=False).head(TOP_N).reset_index(drop=True)
    df_j_top.columns = ['Journal','Publications','SNIP','SJR']
    TC = 6
    ws_jour.set_column(TC, TC, 52); ws_jour.set_column(TC+1, TC+1, 14)
    ws_jour.set_column(TC+2, TC+2, 10); ws_jour.set_column(TC+3, TC+3, 10)
    ws_jour.merge_range(0, TC, 0, TC+3, f'Top {TOP_N} Journals by Publication Count', ri_wide_fmt)
    for ci, cn in enumerate(['Journal','Publications','SNIP','SJR']):
        ws_jour.write(1, TC+ci, cn, hdr_fmt)
    for ri2, row in df_j_top.iterrows():
        sf = stripe_fmt     if ri2 % 2 == 0 else cell_fmt
        sn = stripe_num_fmt if ri2 % 2 == 0 else num_fmt
        ws_jour.write(2+ri2, TC,   str(row['Journal']),      sf)
        ws_jour.write(2+ri2, TC+1, int(row['Publications']), sn)
        for oi, col in enumerate(['SNIP','SJR']):
            v = row[col]
            sd = wb.add_format({'border':1,'num_format':'0.000','valign':'vcenter','bg_color':'#EAF0F7'}) if ri2%2==0 else dec_fmt
            ws_jour.write(2+ri2, TC+2+oi,
                          round(float(v),3) if pd.notna(v) else 'N/A',
                          sd if pd.notna(v) else sf)

    # ── Fields of Research ────────────────────────────────
    ws_for = writer.sheets['Fields of Research']
    df_fc  = df_for[['Field','Count']].copy()
    df_fc['Count']   = pd.to_numeric(df_fc['Count'], errors='coerce')
    df_fc            = df_fc.dropna(subset=['Count'])
    df_fc['ndigits'] = df_fc['Field'].apply(
        lambda x: len(re.match(r'^(\d+)', str(x)).group(1))
        if re.match(r'^(\d+)', str(x)) else 0)
    df_2d = df_fc[df_fc['ndigits']==2].sort_values('Count', ascending=False).head(TOP_N).reset_index(drop=True)
    df_4d = df_fc[df_fc['ndigits']==4].sort_values('Count', ascending=False).head(TOP_N).reset_index(drop=True)
    if not df_2d.empty:
        bar_chart(ws_for, 'Fields of Research',
                  df_2d['Field'].tolist()[::-1], df_2d['Count'].tolist()[::-1],
                  f'Top {TOP_N} 2-Digit Fields of Research',
                  anchor_row=1, anchor_col=4, data_col=60,
                  x_title='No. Publications', chart_w=520, chart_h=380)
    if not df_4d.empty:
        bar_chart(ws_for, 'Fields of Research',
                  df_4d['Field'].tolist()[::-1], df_4d['Count'].tolist()[::-1],
                  f'Top {TOP_N} 4-Digit Fields of Research',
                  anchor_row=1, anchor_col=13, data_col=75,
                  x_title='No. Publications', chart_w=520, chart_h=380)

    # ── Collaboration Summary ─────────────────────────────
    ws_collab = writer.sheets['Collaboration Summary']
    if not df_collab.empty:
        collab_order = ['International Collaboration', 'Only national collaboration',
                        'Only institutional collaboration', 'Single authorship (no collaboration)']
        df_collab_ord = (df_collab.set_index('Collaboration_Type')
                         .reindex(collab_order).fillna(0).reset_index())
        col_chart(ws_collab, 'Collaboration Summary',
                  df_collab_ord['Collaboration_Type'].tolist(),
                  df_collab_ord['Count'].tolist(),
                  'Collaboration Type (Geographic)',
                  anchor_row=len(df_collab)+2, anchor_col=0, data_col=4,
                  x_title='Collaboration Type', y_title='No. Publications',
                  chart_w=540, chart_h=340)

    # ── Country Collaboration ─────────────────────────────
    ws_cc = writer.sheets['Country Collaboration']
    ws_cc.set_column(0, 0, 30); ws_cc.set_column(1, 1, 16)
    ws_cc.write(0, 0, f'Country (excl. {HOME_COUNTRY})', hdr_fmt)
    ws_cc.write(0, 1, 'Publications', hdr_fmt)
    for ri_c, row in df_country_collab.iterrows():
        sf = stripe_fmt     if ri_c % 2 == 0 else cell_fmt
        sn = stripe_num_fmt if ri_c % 2 == 0 else num_fmt
        ws_cc.write(ri_c+1, 0, str(row['Country']),      sf)
        ws_cc.write(ri_c+1, 1, int(row['Publications']), sn)
    df_cc_top = df_country_collab.head(20)
    if not df_cc_top.empty:
        bar_chart(ws_cc, 'Country Collaboration',
                  df_cc_top['Country'].tolist()[::-1],
                  df_cc_top['Publications'].tolist()[::-1],
                  f'Top 20 Collaborating Countries (excl. {HOME_COUNTRY})',
                  anchor_row=1, anchor_col=4, data_col=20,
                  x_title='No. Publications', chart_w=540, chart_h=520)

    # ── Org Type Collaboration ────────────────────────────
    ws_org   = writer.sheets['Org Type Collaboration']
    df_org_c = df_org_collab_summary[
        df_org_collab_summary['Organisation Type'].notna() &
        (df_org_collab_summary['Organisation Type'].str.strip() != '')].copy()
    df_org_c['Publications'] = pd.to_numeric(df_org_c['Publications'], errors='coerce')
    df_org_c = df_org_c.dropna(subset=['Publications'])
    if not df_org_c.empty:
        col_chart(ws_org, 'Org Type Collaboration',
                  df_org_c['Organisation Type'].tolist(),
                  df_org_c['Publications'].tolist(),
                  'Publications by Organisation Type',
                  anchor_row=len(df_org_c)+2, anchor_col=0, data_col=4,
                  x_title='Organisation Type', y_title='No. Publications',
                  chart_w=520, chart_h=340)

    # ── Industry Co-authorship ────────────────────────────
    ws_ind = writer.sheets['Industry Co-authorship']
    ws_ind.set_column(0, 0, 52); ws_ind.set_column(1, 1, 24); ws_ind.set_column(2, 2, 18)
    ws_ind.write(0, 0, 'Organisation',              hdr_fmt)
    ws_ind.write(0, 1, 'Co-authored Publications',  hdr_fmt)
    ws_ind.write(0, 2, 'Total Citations',            hdr_fmt)
    for i, row in df_industry_orgs.iterrows():
        sf = stripe_fmt     if i % 2 == 0 else cell_fmt
        sn = stripe_num_fmt if i % 2 == 0 else num_fmt
        ws_ind.write(i+1, 0, str(row["Organisation"]),             sf)
        ws_ind.write(i+1, 1, int(row["Co-authored Publications"]), sn)
        ws_ind.write(i+1, 2, int(row["Total Citations"]),          sn)
    df_ind_top10 = df_industry_orgs.head(TOP_N)
    if not df_ind_top10.empty:
        bar_chart(ws_ind, 'Industry Co-authorship',
                  df_ind_top10["Organisation"].tolist()[::-1],
                  df_ind_top10["Co-authored Publications"].tolist()[::-1],
                  f'Top {TOP_N} Industry Co-authoring Organisations',
                  anchor_row=1, anchor_col=4, data_col=20,
                  x_title='Co-authored Publications', chart_w=520, chart_h=400)
    if not df_ind_pubs.empty:
        pub_start = len(df_industry_orgs) + 3
        ws_ind.set_column(0, 0, 60); ws_ind.set_column(1, 1, 22)
        ws_ind.set_column(2, 2, 6);  ws_ind.set_column(3, 3, 14)
        ws_ind.set_column(4, 4, 36); ws_ind.set_column(5, 5, 12)
        ws_ind.merge_range(pub_start, 0, pub_start, 5,
                           f'Top {len(df_ind_pubs)} Co-authored Publications by Citations', ri_wide_fmt)
        for ci, cn in enumerate(['Title','DOI','Year','Type','Co-authoring Organisations','Citations']):
            ws_ind.write(pub_start+1, ci, cn, hdr_fmt)
        url_fmt_i    = wb.add_format({'border':1,'font_color':'#0563C1','underline':True,
                                      'valign':'vcenter','text_wrap':True})
        url_stripe_i = wb.add_format({'border':1,'font_color':'#0563C1','underline':True,
                                      'valign':'vcenter','text_wrap':True,'bg_color':'#EAF0F7'})
        for ri_i, row in df_ind_pubs.iterrows():
            sf  = url_stripe_i   if ri_i % 2 == 0 else url_fmt_i
            sc  = stripe_fmt     if ri_i % 2 == 0 else cell_fmt
            sn  = stripe_num_fmt if ri_i % 2 == 0 else num_fmt
            doi = str(row['DOI']); url = f'https://doi.org/{doi}' if doi and doi != 'nan' else ''
            if url: ws_ind.write_url(pub_start+2+ri_i, 0, url, sf, str(row['Title']))
            else:   ws_ind.write(pub_start+2+ri_i, 0, str(row['Title']), sc)
            ws_ind.write(pub_start+2+ri_i, 1, doi if doi != 'nan' else '', sc)
            ws_ind.write(pub_start+2+ri_i, 2, int(row['Year']) if pd.notna(row['Year']) else '', sc)
            ws_ind.write(pub_start+2+ri_i, 3, str(row['Type']), sc)
            ws_ind.write(pub_start+2+ri_i, 4, str(row['Co-authoring Organisations']), sc)
            ws_ind.write(pub_start+2+ri_i, 5, int(row['Citations']), sn)
            ws_ind.set_row(pub_start+2+ri_i, 36)
    print("  ✅ Industry Co-authorship tab written.")

    # ── Funders Summary ───────────────────────────────────
    ws_fund = writer.sheets['Funders Summary']
    ws_fund.set_column(0, 0, 52); ws_fund.set_column(1, 1, 16); ws_fund.set_column(2, 2, 16)
    ws_fund.write(0, 0, 'Funder',          hdr_fmt)
    ws_fund.write(0, 1, 'Publications',    hdr_fmt)
    ws_fund.write(0, 2, 'Total Citations', hdr_fmt)
    for i, row in df_funders_summary.iterrows():
        sf = stripe_fmt     if i % 2 == 0 else cell_fmt
        sn = stripe_num_fmt if i % 2 == 0 else num_fmt
        ws_fund.write(i+1, 0, str(row['Funder']),          sf)
        ws_fund.write(i+1, 1, int(row['Publications']),    sn)
        ws_fund.write(i+1, 2, int(row['Total Citations']), sn)
    df_fund_top = df_funders_summary.head(TOP_N).reset_index(drop=True)
    FC = 4
    ws_fund.set_column(FC, FC, 52); ws_fund.set_column(FC+1, FC+1, 16); ws_fund.set_column(FC+2, FC+2, 16)
    ws_fund.merge_range(0, FC, 0, FC+2, f'Top {TOP_N} Funders by Publication Count', ri_wide_fmt)
    for ci, cn in enumerate(['Funder', 'Publications', 'Total Citations']):
        ws_fund.write(1, FC+ci, cn, hdr_fmt)
    for ri_f, row in df_fund_top.iterrows():
        sf = stripe_fmt     if ri_f % 2 == 0 else cell_fmt
        sn = stripe_num_fmt if ri_f % 2 == 0 else num_fmt
        ws_fund.write(2+ri_f, FC,   str(row['Funder']),          sf)
        ws_fund.write(2+ri_f, FC+1, int(row['Publications']),    sn)
        ws_fund.write(2+ri_f, FC+2, int(row['Total Citations']), sn)
    bar_chart(ws_fund, 'Funders Summary',
              df_fund_top['Funder'].tolist()[::-1],
              df_fund_top['Publications'].tolist()[::-1],
              f'Top {TOP_N} Funders by Publication Count',
              anchor_row=TOP_N+4, anchor_col=FC, data_col=20,
              x_title='No. Publications', chart_w=560, chart_h=400)
    print("  ✅ Funders Summary tab written.")

    # ── Policy Publishers ─────────────────────────────────
    ws_pol  = writer.sheets['Policy Publishers']
    df_pol  = df_policy_publishers.copy()
    df_pol['Publications Cited'] = pd.to_numeric(df_pol['Publications Cited'], errors='coerce')
    df_pol['Policy Documents']   = pd.to_numeric(df_pol['Policy Documents'],   errors='coerce')
    df_pol_top = (df_pol.dropna(subset=['Publications Cited'])
                  .sort_values('Publications Cited', ascending=False)
                  .head(TOP_N)[['Publisher','Policy Documents','Publications Cited']]
                  .reset_index(drop=True))
    PC = 4
    ws_pol.set_column(PC, PC, 55); ws_pol.set_column(PC+1, PC+1, 18); ws_pol.set_column(PC+2, PC+2, 20)
    ws_pol.merge_range(0, PC, 0, PC+2,
                       f'Top {min(TOP_N, len(df_pol_top))} Policy Publishers by Publications Cited', ri_wide_fmt)
    for ci, cn in enumerate(['Publisher','Policy Documents','Publications Cited']):
        ws_pol.write(1, PC+ci, cn, hdr_fmt)
    for ri3, row in df_pol_top.iterrows():
        sf = stripe_fmt     if ri3 % 2 == 0 else cell_fmt
        sn = stripe_num_fmt if ri3 % 2 == 0 else num_fmt
        ws_pol.write(2+ri3, PC,   str(row['Publisher']),          sf)
        ws_pol.write(2+ri3, PC+1, int(row['Policy Documents']),   sn)
        ws_pol.write(2+ri3, PC+2, int(row['Publications Cited']), sn)
    df_top_pol_pubs = (df_up[df_up['policy_citations'] > 0][['title','doi','policy_citations']]
                       .sort_values('policy_citations', ascending=False)
                       .head(TOP_N).reset_index(drop=True))
    if not df_top_pol_pubs.empty:
        pp_start = len(df_pol_top) + 4
        ws_pol.set_column(PC, PC, 70); ws_pol.set_column(PC+1, PC+1, 18)
        ws_pol.merge_range(pp_start, PC, pp_start, PC+1,
                           f'Top {len(df_top_pol_pubs)} Publications by Policy Citations', ri_wide_fmt)
        ws_pol.write(pp_start+1, PC,   'Publication Title', hdr_fmt)
        ws_pol.write(pp_start+1, PC+1, 'Policy Citations',  hdr_fmt)
        url_fmt = wb.add_format({'border': 1, 'font_color': '#0563C1', 'underline': True,
                                 'valign': 'vcenter', 'text_wrap': True})
        url_stripe_fmt = wb.add_format({'border': 1, 'font_color': '#0563C1', 'underline': True,
                                        'valign': 'vcenter', 'text_wrap': True, 'bg_color': '#EAF0F7'})
        for ri4, row in df_top_pol_pubs.iterrows():
            sf  = url_stripe_fmt if ri4 % 2 == 0 else url_fmt
            sn  = stripe_num_fmt if ri4 % 2 == 0 else num_fmt
            doi = str(row['doi']); url = f'https://doi.org/{doi}' if doi and doi != 'nan' else ''
            if url: ws_pol.write_url(pp_start+2+ri4, PC, url, sf, str(row['title']))
            else:   ws_pol.write(pp_start+2+ri4, PC, str(row['title']), stripe_fmt if ri4 % 2 == 0 else cell_fmt)
            ws_pol.write(pp_start+2+ri4, PC+1, int(row['policy_citations']), sn)
            ws_pol.set_row(pp_start+2+ri4, 30)

    # ── Patents Summary ───────────────────────────────────
    ws_pat = writer.sheets['Patents Summary']
    df_top_pat_pubs = (df_up[df_up['patent_citations'] > 0][['title','doi','patent_citations']]
                       .sort_values('patent_citations', ascending=False)
                       .head(TOP_N).reset_index(drop=True))
    if not df_top_pat_pubs.empty:
        pat_tbl_col = 6
        ws_pat.set_column(pat_tbl_col, pat_tbl_col, 70)
        ws_pat.set_column(pat_tbl_col+1, pat_tbl_col+1, 18)
        ws_pat.merge_range(1, pat_tbl_col, 1, pat_tbl_col+1,
                           f'Top {len(df_top_pat_pubs)} Publications by Patent Citations', ri_wide_fmt)
        ws_pat.write(2, pat_tbl_col,   'Publication Title', hdr_fmt)
        ws_pat.write(2, pat_tbl_col+1, 'Patent Citations',  hdr_fmt)
        url_fmt_p    = wb.add_format({'border': 1, 'font_color': '#0563C1', 'underline': True, 'valign': 'vcenter', 'text_wrap': True})
        url_stripe_p = wb.add_format({'border': 1, 'font_color': '#0563C1', 'underline': True, 'valign': 'vcenter', 'text_wrap': True, 'bg_color': '#EAF0F7'})
        for ri5, row in df_top_pat_pubs.iterrows():
            sf = url_stripe_p   if ri5 % 2 == 0 else url_fmt_p
            sn = stripe_num_fmt if ri5 % 2 == 0 else num_fmt
            doi = str(row['doi']); url = f'https://doi.org/{doi}' if doi and doi != 'nan' else ''
            if url: ws_pat.write_url(3+ri5, pat_tbl_col, url, sf, str(row['title']))
            else:   ws_pat.write(3+ri5, pat_tbl_col, str(row['title']), stripe_fmt if ri5 % 2 == 0 else cell_fmt)
            ws_pat.write(3+ri5, pat_tbl_col+1, int(row['patent_citations']), sn)
            ws_pat.set_row(3+ri5, 30)

    # ── Clinical Trials ───────────────────────────────────
    ws_ct = writer.sheets['Clinical Trials Summary']
    df_top_ct_pubs = (df_up[df_up['clinical_citations'] > 0][['title','doi','clinical_citations']]
                      .sort_values('clinical_citations', ascending=False)
                      .head(TOP_N).reset_index(drop=True))
    if not df_top_ct_pubs.empty:
        ct_tbl_col = 6
        ws_ct.set_column(ct_tbl_col, ct_tbl_col, 70)
        ws_ct.set_column(ct_tbl_col+1, ct_tbl_col+1, 22)
        ws_ct.merge_range(1, ct_tbl_col, 1, ct_tbl_col+1,
                          f'Top {len(df_top_ct_pubs)} Publications by Clinical Trial Citations', ri_wide_fmt)
        ws_ct.write(2, ct_tbl_col,   'Publication Title',        hdr_fmt)
        ws_ct.write(2, ct_tbl_col+1, 'Clinical Trial Citations', hdr_fmt)
        url_fmt_c    = wb.add_format({'border': 1, 'font_color': '#0563C1', 'underline': True, 'valign': 'vcenter', 'text_wrap': True})
        url_stripe_c = wb.add_format({'border': 1, 'font_color': '#0563C1', 'underline': True, 'valign': 'vcenter', 'text_wrap': True, 'bg_color': '#EAF0F7'})
        for ri6, row in df_top_ct_pubs.iterrows():
            sf = url_stripe_c   if ri6 % 2 == 0 else url_fmt_c
            sn = stripe_num_fmt if ri6 % 2 == 0 else num_fmt
            doi = str(row['doi']); url = f'https://doi.org/{doi}' if doi and doi != 'nan' else ''
            if url: ws_ct.write_url(3+ri6, ct_tbl_col, url, sf, str(row['title']))
            else:   ws_ct.write(3+ri6, ct_tbl_col, str(row['title']), stripe_fmt if ri6 % 2 == 0 else cell_fmt)
            ws_ct.write(3+ri6, ct_tbl_col+1, int(row['clinical_citations']), sn)
            ws_ct.set_row(3+ri6, 30)

    # ── Altmetric Attention ───────────────────────────────
    ws_alt = writer.sheets['Altmetric Attention']
    ws_alt.set_column(0, 0, 70); ws_alt.set_column(1, 1, 30)
    ws_alt.set_column(2, 2, 8);  ws_alt.set_column(3, 3, 14); ws_alt.set_column(4, 4, 16)
    for ci, cn in enumerate(['Publication Title', 'DOI', 'Year', 'Type', 'Altmetric Score']):
        ws_alt.write(0, ci, cn, hdr_fmt)
    url_fmt_a    = wb.add_format({'border': 1, 'font_color': '#0563C1', 'underline': True, 'valign': 'vcenter', 'text_wrap': True})
    url_stripe_a = wb.add_format({'border': 1, 'font_color': '#0563C1', 'underline': True, 'valign': 'vcenter', 'text_wrap': True, 'bg_color': '#EAF0F7'})
    num_fmt_a    = wb.add_format({'border': 1, 'num_format': '#,##0', 'valign': 'vcenter'})
    num_stripe_a = wb.add_format({'border': 1, 'num_format': '#,##0', 'valign': 'vcenter', 'bg_color': '#EAF0F7'})
    cell_fmt_c   = wb.add_format({'border': 1, 'valign': 'vcenter', 'align': 'center'})
    cell_stripe_c= wb.add_format({'border': 1, 'valign': 'vcenter', 'align': 'center', 'bg_color': '#EAF0F7'})
    for ri, row in df_altmetric_top.iterrows():
        ws_alt.set_row(ri + 1, 36)
        sf  = url_stripe_a  if ri % 2 == 0 else url_fmt_a
        sc  = cell_stripe_c if ri % 2 == 0 else cell_fmt_c
        sn  = num_stripe_a  if ri % 2 == 0 else num_fmt_a
        scf = stripe_fmt    if ri % 2 == 0 else cell_fmt
        doi = str(row['DOI']); url = f'https://doi.org/{doi}' if doi and doi != 'nan' else ''
        if url: ws_alt.write_url(ri+1, 0, url, sf, str(row['Title']))
        else:   ws_alt.write(ri+1, 0, str(row['Title']), scf)
        ws_alt.write(ri+1, 1, doi if doi != 'nan' else '', scf)
        ws_alt.write(ri+1, 2, int(row['Year']) if pd.notna(row['Year']) else '', sc)
        ws_alt.write(ri+1, 3, str(row['Type']), scf)
        ws_alt.write(ri+1, 4, int(row['Altmetric Score']), sn)

    # ── Top Researchers by Impact ─────────────────────────
    ws_tri    = writer.sheets['Top Researchers by Impact']
    NAVY_R    = '#1F3864'
    BLUE_R    = '#1F497D'
    tri_title_fmt = wb.add_format({'bold': True, 'font_size': 13, 'font_color': NAVY_R, 'bottom': 2, 'bottom_color': BLUE_R})
    tri_sec_fmt   = wb.add_format({'bold': True, 'font_size': 11, 'font_color': 'white', 'bg_color': NAVY_R, 'border': 1})
    tri_hdr_fmt   = wb.add_format({'bold': True, 'font_size': 10, 'font_color': 'white', 'bg_color': BLUE_R, 'border': 1})
    tri_str_fmt   = wb.add_format({'border': 1, 'font_size': 9, 'valign': 'vcenter'})
    tri_num_fmt   = wb.add_format({'border': 1, 'font_size': 9, 'num_format': '#,##0', 'align': 'right', 'valign': 'vcenter'})
    tri_pct_fmt   = wb.add_format({'border': 1, 'font_size': 9, 'align': 'right', 'valign': 'vcenter'})
    tri_stripe    = wb.add_format({'border': 1, 'font_size': 9, 'valign': 'vcenter', 'bg_color': '#EAF0F7'})
    tri_stripe_n  = wb.add_format({'border': 1, 'font_size': 9, 'num_format': '#,##0', 'align': 'right', 'valign': 'vcenter', 'bg_color': '#EAF0F7'})
    tri_stripe_p  = wb.add_format({'border': 1, 'font_size': 9, 'align': 'right', 'valign': 'vcenter', 'bg_color': '#EAF0F7'})
    ws_tri.set_column(0, 0, 6); ws_tri.set_column(1, 1, 35); ws_tri.set_column(2, 2, 26)

    def _write_section(row, title):
        ws_tri.merge_range(row, 0, row, 2, title, tri_sec_fmt)
        ws_tri.set_row(row, 22); return row + 1

    def _write_headers(row, col3_label):
        ws_tri.write(row, 0, '#',          tri_hdr_fmt)
        ws_tri.write(row, 1, 'Researcher', tri_hdr_fmt)
        ws_tri.write(row, 2, col3_label,   tri_hdr_fmt)
        ws_tri.set_row(row, 18); return row + 1

    def _write_rows(row, df, val_col):
        for ri, (_, r) in enumerate(df.iterrows()):
            bg_s = tri_stripe   if ri % 2 == 0 else tri_str_fmt
            bg_n = tri_stripe_n if ri % 2 == 0 else tri_num_fmt
            ws_tri.write(row, 0, ri + 1,               bg_s)
            ws_tri.write(row, 1, str(r['Researcher']), bg_s)
            ws_tri.write(row, 2, int(r[val_col]),      bg_n)
            ws_tri.set_row(row, 16); row += 1
        return row + 1

    ws_tri.merge_range(0, 0, 0, 2, f'Top Researchers by Impact — {INSTITUTION}, {FACULTY}', tri_title_fmt)
    ws_tri.set_row(0, 28)
    row_t = 2
    for section_title, df_t, val_col in [
        ('Most Policy Citations',        top_res_policy,    'Policy Citations'),
        ('Most Patent Citations',         top_res_patent,    'Patent Citations'),
        ('Most Clinical Trial Citations', top_res_clinical,  'Clinical Trial Citations'),
        ('Most Publications',             top_res_pubs,      'Total Publications'),
        ('Most Cited Researchers',        top_res_cited,     'Total Citations'),
    ]:
        if df_t.empty: continue
        row_t = _write_section(row_t, section_title)
        row_t = _write_headers(row_t, val_col)
        row_t = _write_rows(row_t, df_t, val_col)

    row_t = _write_section(row_t, 'ORCID Adoption')
    ws_tri.write(row_t, 0, 'Metric',           tri_hdr_fmt)
    ws_tri.write(row_t, 1, 'Count',            tri_hdr_fmt)
    ws_tri.write(row_t, 2, '% of Researchers', tri_hdr_fmt)
    ws_tri.set_row(row_t, 18); row_t += 1
    for ri, (label, count, pct) in enumerate([
        ('Total researchers',         total_researchers, '100%'),
        ('Researchers with an ORCID', with_orcid,        f'{round(with_orcid/total_researchers*100,1)}%'),
        ('Researchers with no ORCID', no_orcid,          f'{round(no_orcid/total_researchers*100,1)}%'),
        ('Researchers with 2 ORCIDs', with_two_orcid,    f'{round(with_two_orcid/total_researchers*100,1)}%'),
    ]):
        bg_s = tri_stripe   if ri % 2 == 0 else tri_str_fmt
        bg_n = tri_stripe_n if ri % 2 == 0 else tri_num_fmt
        bg_p = tri_stripe_p if ri % 2 == 0 else tri_pct_fmt
        ws_tri.write(row_t, 0, label, bg_s)
        ws_tri.write(row_t, 1, count, bg_n)
        ws_tri.write(row_t, 2, pct,   bg_p)
        ws_tri.set_row(row_t, 16); row_t += 1

    print("  ✅ Top Researchers by Impact tab written.")
    print(f"  📊 All charts and formatted tables embedded.")

files.download(p1_file)
print(f"\n✅ Complete: {p1_file}")
print("\nReport tabs:")
print("  • Report Information          ← date, institution, faculty, period, methodology, tab guide")
print("  • Researchers")
print("  • Publications Summary        ← pubs by year + pubs by type charts")
print("  • Journals Summary            ← formatted top-10 table with striping")
print("  • Fields of Research          ← 2-digit bar + 4-digit bar")
print("  • Collaboration Summary       ← vertical bar chart")
print("  • Country Collaboration       ← formatted table + top 20 bar chart")
print("  • Org Type Collaboration      ← vertical bar chart")
print("  • Industry Co-authorship      ← org table + bar chart + top cited pubs")
print("  • Funders Summary             ← full table + top-10 formatted table + bar chart")
print("  • Policy Publishers           ← formatted top-10 table + top cited pubs")
print("  • Patents Summary             ← assignee table + top cited pubs")
print("  • Clinical Trials Summary     ← sponsor table + top cited pubs")
print("  • Complete Publications       ← full pub-level data incl. researchers column")
print("  • Altmetric Attention         ← top 10 by Altmetric score, hyperlinked")
print("  • Top Researchers by Impact   ← policy, patent, clinical, pubs, citations, ORCID")


Setting up environment...

--- STEP 1: DIMENSIONS LOGIN ---
Please paste your Dimensions API Key: 8371E0EF67F745AA93E5204C2E71C635


Using default endpoint: 'https://app.dimensions.ai'


Dimcli - Dimensions API Client (v1.6)
Connected to: <https://app.dimensions.ai/api/dsl> - DSL v2.14
Method: manual login
✅ Connected.

--- STEP 2: INPUT PARAMETERS ---
Start Year (e.g., 2020): 2020
End Year (e.g., 2025): 2025
Faculty / Department name (e.g., Faculty of Science): Faculty X
Institution GRID ID (e.g., grid.1005.4): 
  Looking up institution name for ...
Returned Organizations: 0
Time: 0.33s
  ⚠️  Could not find GRID ID ''.
  Please enter institution name manually: Lilliput University

  Institution : Lilliput University
  Faculty     : Faculty X
  Period      : 2020 – 2025

Please upload your researcher ID list (CSV or Excel):


Saving Lilliput IDs.xlsx to Lilliput IDs (3).xlsx
✅ Loaded 200 researcher IDs.

--- STEP 3: FETCHING DATA ---

[1/5] Fetching researcher metadata...
Returned Researchers: 20 (total = 25)
Time: 0.39s
Returned Researchers: 20 (total = 25)
Time: 7.52s
Returned Researchers: 20 (total = 25)
Time: 3.90s
Returned Researchers: 20 (total = 25)
Time: 0.34s
Returned Researchers: 20 (total = 25)
Time: 0.31s
Returned Researchers: 20 (total = 25)
Time: 5.74s
Returned Researchers: 20 (total = 25)
Time: 5.70s
Returned Researchers: 20 (total = 25)
Time: 0.33s
  ✅ Metadata: done.                    

[2/5] Fetching publications (paginated, includes citations field)...
Returned Publications: 1000 (total = 2820)
Time: 7.16s
Returned Publications: 1000 (total = 2820)
Time: 10.36s
Returned Errors: 1
Time: 16.20s
Service unavailable
1 EvaluationError found

Your query is too long e.g. because it includes too many filters or search words. Please review it by keeping in mind the guidelines on https://docs.dime

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Complete: Researcher_Metrics_Report_2020_2025.xlsx

Report tabs:
  • Report Information          ← date, institution, faculty, period, methodology, tab guide
  • Researchers
  • Publications Summary        ← pubs by year + pubs by type charts
  • Journals Summary            ← formatted top-10 table with striping
  • Fields of Research          ← 2-digit bar + 4-digit bar
  • Collaboration Summary       ← vertical bar chart
  • Country Collaboration       ← formatted table + top 20 bar chart
  • Org Type Collaboration      ← vertical bar chart
  • Industry Co-authorship      ← org table + bar chart + top cited pubs
  • Funders Summary             ← full table + top-10 formatted table + bar chart
  • Policy Publishers           ← formatted top-10 table + top cited pubs
  • Patents Summary             ← assignee table + top cited pubs
  • Clinical Trials Summary     ← sponsor table + top cited pubs
  • Complete Publications       ← full pub-level data incl. researchers column
  • Altme